<a href="https://colab.research.google.com/github/WenqiaoDuan/Econ5200-replication-template/blob/main/01_positioning_and_replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [3]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [4]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    # TODO: read the file(s), e.g. df = pd.read_stata(paths[0])
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

downloading https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta
  data/lmb-data.dta  7.2 MB
Lee (2008), incumbency advantage in US House elections


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1.
Caughey and Sekhon (2011) questioned local randomization in close U.S. House elections, finding systematic imbalances in candidates' pre-election characteristics near the threshold.

2.
Eggers et al. (2015), analyzing over 40,000 close elections across multiple countries, found that these anomalies were largely specific to U.S. House elections rather than a general feature of electoral RDD.

3.
de la Cuesta and Imai (2016) argued that local randomization requires stronger assumptions than conventional RDD continuity. They also found that evidence of sorting weakened after accounting for multiple testing and was sensitive to estimation methods.

**The identification strategy and its assumption, in my own words:**

Lee (2008) used a sharp regression discontinuity design (RDD) to estimate the causal effect of party incumbency in U.S. House elections. The running variable is the Democratic vote margin over its strongest opponent, with a threshold of zero. Treatment occurs when the Democratic candidate wins the election. The outcomes include the Democratic vote share and the probability of winning in the next election.
The main assumption is that potential outcomes would change smoothly at the cutoff without treatment. If this holds, barely winning and barely losing districts can be compared to estimate the effect of incumbency.

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

In [10]:
# GIVEN — setup: the two RD packages Colab does not have, then the libraries
%pip install rdrobust==2.0.0 rddensity==3.0 -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
from rdrobust import rdrobust
from rddensity import rddensity

print("Setup complete.")



Setup complete.


In [11]:
# GIVEN — Section 2 starts here: load the data and build the running variable

lmb = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/lmb-data.dta")

n_before = len(lmb)

lmb = lmb.dropna(subset=["demvoteshare", "lagdemvoteshare"]).copy()

print(f"Original observations: {n_before:,}")
print(f"Observations removed: {n_before - len(lmb):,}")
print(f"Observations after cleaning: {len(lmb):,}")

lmb["margin"] = lmb["lagdemvoteshare"] - 0.5
CUTOFF = 0.0

print(f"Exact duplicate rows: {lmb.duplicated().sum()}")

print(f"{len(lmb):,} district-years, {lmb['year'].min():.0f}-{lmb['year'].max():.0f}")

lmb[["demvoteshare", "lagdemvoteshare", "margin", "democrat", "lagdemocrat"]].describe().round(3)

Original observations: 13,588
Observations removed: 22
Observations after cleaning: 13,566
Exact duplicate rows: 0
13,566 district-years, 1948-1990


,demvoteshare,lagdemvoteshare,margin,democrat,lagdemocrat
count,13566.000,13566.000,13566.000,13566.000,13566.000
mean,0.582,0.577,0.077,0.597,0.582
std,0.230,0.226,0.226,0.491,0.493
min,0.000,0.000,-0.500,0.000,0.000
25%,0.416,0.411,-0.089,0.000,0.000
50%,0.557,0.546,0.046,1.000,1.000
75%,0.720,0.712,0.212,1.000,1.000
max,1.000,1.000,0.500,1.000,1.000


In [12]:
# YOUR TURN — Section 2, no AI: set H, KERNEL, ORDER, fit rd_fit, and set JUMP

# RD specification
H = 0.5
KERNEL = "rectangular"
ORDER = 4

# Keep observations within the bandwidth
sample = lmb.loc[
    lmb["margin"].notna()
    & lmb["demvoteshare"].notna()
    & (lmb["margin"].abs() <= H)
].copy()

# Running variable and treatment
sample["x"] = sample["margin"]
sample["D"] = (sample["x"] >= 0).astype(int)

# Separate quartic polynomial on both sides
X = pd.DataFrame(
    {"D": sample["D"]},
    index=sample.index
)

for k in range(1, ORDER + 1):
    X[f"x{k}"] = sample["x"] ** k
    X[f"D_x{k}"] = sample["D"] * sample["x"] ** k

X = sm.add_constant(X)

# Fit OLS regression
rd_fit = sm.OLS(
    sample["demvoteshare"],
    X
).fit(cov_type="HC1")

# Treatment effect at cutoff
JUMP = float(rd_fit.params["D"])

print(rd_fit.summary())
print(f"\nRD estimate: {JUMP:.6f}")
print(f"Effect in percentage points: {100 * JUMP:.3f}")
print(f"Standard error: {rd_fit.bse['D']:.6f}")
print(f"Observations: {int(rd_fit.nobs)}")

                            OLS Regression Results                            
Dep. Variable:           demvoteshare   R-squared:                       0.717
Model:                            OLS   Adj. R-squared:                  0.717
Method:                 Least Squares   F-statistic:                     3567.
Date:                Tue, 10 Nov 2026   Prob (F-statistic):               0.00
Time:                        02:55:33   Log-Likelihood:                 9272.5
No. Observations:               13566   AIC:                        -1.853e+04
Df Residuals:                   13556   BIC:                        -1.845e+04
Df Model:                           9                                         
Covariance Type:                  HC1                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.4534      0.005     91.631      0.0

### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [15]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

# One line per row; take my_estimate from the cell that computed it, never type it in.
# headline.loc["Table 2, col 3: <what it measures>"] = [<paper's number>, <your variable>, np.nan, "<likely source>"]

headline.loc["Table 2, col 1: effect on Democratic vote share"] = [
    0.077,
    JUMP,
    np.nan,
    "Different sample periods and running-variable definitions"
]

headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

,paper_estimate,my_estimate,gap_pct,likely_source
estimate,,,,
"Table 2, col 1: effect on Democratic vote share",0.077,0.086126,11.851966,Different sample periods and running-variable ...


### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | Lee (2008), Table 2, Column (1): Democratic vote share | 0.077 VS 0.086126 | +0.9126 percentage points (+11.85%) | sample construction / variable definition / software default / undocumented choice |  checked the election years in Python and found that 1952, 1962, 1972, and 1982 are missing from my sample. I also compared the running-variable definitions. However, I could not determine exactly how much these differences explain the gap.| The paper describes its sample and running variable, but the course dataset is different. The exact reason for the gap remains unresolved. |


**Ambiguities in the paper, to take into Section 3:**

- (1)The different years may change which elections enter the regression.;
(2) This may matter for the estimate, but I cannot measure how much of the 0.91 point difference it explains with the data available.; (3) In a two-party race these measures are closely related, so this difference alone does not necessarily explain the gap.